In [0]:
%sql
CREATE OR REPLACE TABLE data_warehouse_factory.gold.fct_final_operator_assignments AS
WITH 
-- 1. Rozbicie wielodniowych planów na pojedyncze dni robocze
expanded_plan_days AS (
  SELECT 
    p.plan_key,
    p.line_code,
    p.cell_code,
    p.cell_name,
    p.start_datetime,
    p.end_datetime,
    CAST(p.start_datetime AS DATE) AS plan_start_date,
    CAST(p.end_datetime AS DATE) AS plan_end_date,
    date_format(p.start_datetime, 'HH:mm:ss') AS plan_start_time_str,
    date_format(p.end_datetime, 'HH:mm:ss') AS plan_end_time_str,
    
    explode(sequence(CAST(p.start_datetime AS DATE), CAST(p.end_datetime AS DATE), interval 1 day)) AS production_date
  FROM data_warehouse_factory.silver.silver_production_plan p
  WHERE p.cell_code IS NOT NULL AND p.cell_code != 'NO_CELL'
),

-- 2. Wyznaczenie dziennych ram czasowych planu
daily_plan_windows AS (
  SELECT 
    plan_key,
    line_code,
    cell_code,
    cell_name,
    production_date,
    CAST(date_format(production_date, 'yyyyMMdd') AS INT) AS date_key,
    
    to_timestamp(concat(cast(production_date as string), ' ', 
      CASE WHEN production_date = plan_start_date THEN plan_start_time_str ELSE '06:00:00' END
    )) AS daily_plan_start,
    
    to_timestamp(concat(cast(production_date as string), ' ', 
      CASE WHEN production_date = plan_end_date THEN plan_end_time_str ELSE '14:00:00' END
    )) AS daily_plan_end
  FROM expanded_plan_days
),

-- 3. Złączenie planu z domyślną obsadą SCD-2
planned_cells AS (
  SELECT
    dp.plan_key,
    dp.production_date AS start_date,
    dp.date_key,
    dp.line_code,
    dp.cell_code,
    dp.cell_name,
    dp.daily_plan_start AS plan_start,
    dp.daily_plan_end AS plan_end,
    a.default_employee_key,
    a.backup_employee_key
  FROM daily_plan_windows dp
  LEFT JOIN data_warehouse_factory.gold.dim_employee_assignments a 
    ON dp.cell_code = a.cell_code 
    AND dp.production_date BETWEEN a.valid_from AND a.valid_to
  WHERE dp.daily_plan_start < dp.daily_plan_end
),

-- 4. SWAP-IN: Kto został przeniesiony NA daną komórkę
fct_swap_roles_in AS (
  SELECT
    e.production_date AS start_date,
    e.cell_code,
    e.daily_event_start AS start_ts,
    e.daily_event_end AS end_ts,
    e.employee_key
  FROM data_warehouse_factory.gold.fct_events_daily e
  INNER JOIN data_warehouse_factory.gold.dim_events d
    ON upper(trim(e.event_type)) = upper(trim(d.event_type_name))
  WHERE d.is_swap = TRUE AND e.cell_code IS NOT NULL AND e.cell_code != 'NO ASSIGNMENT'
),

-- 5. SWAP-OUT: Pracownicy oddelegowani (niedostępni na swojej komórce macierzystej)
fct_swap_roles_out AS (
  SELECT
    e.production_date AS start_date,
    e.employee_key AS swapped_out_operator_key,
    e.daily_event_start AS start_ts,
    e.daily_event_end AS end_ts
  FROM data_warehouse_factory.gold.fct_events_daily e
  INNER JOIN data_warehouse_factory.gold.dim_events d
    ON upper(trim(e.event_type)) = upper(trim(d.event_type_name))
  WHERE d.is_swap = TRUE
),

-- 6. Nieobecności (L4, urlopy itp.)
absences AS (
  SELECT 
    e.production_date AS start_date,
    e.employee_key AS absent_operator_key,
    e.daily_event_start AS start_ts,
    e.daily_event_end AS end_ts
  FROM data_warehouse_factory.gold.fct_events_daily e
  INNER JOIN data_warehouse_factory.gold.dim_events d
    ON upper(trim(e.event_type)) = upper(trim(d.event_type_name))
  WHERE d.is_absence = TRUE
),

-- 7. Punkty graniczne (Breakpoints) per dzień i komórka
all_time_points AS (
  -- Granice planu
  SELECT start_date, cell_code, plan_start AS bp_time FROM planned_cells
  UNION DISTINCT
  SELECT start_date, cell_code, plan_end AS bp_time FROM planned_cells
  
  -- Punkty ze Swap-In (ktoś przychodzi na tę komórkę)
  UNION DISTINCT
  SELECT s.start_date, s.cell_code, s.start_ts AS bp_time FROM fct_swap_roles_in s
  UNION DISTINCT
  SELECT s.start_date, s.cell_code, s.end_ts AS bp_time FROM fct_swap_roles_in s
  
  -- Punkty ze Swap-Out (domyślny operator tej komórki został wysłany gdzie indziej)
  UNION DISTINCT
  SELECT p.start_date, p.cell_code, so.start_ts AS bp_time
  FROM planned_cells p
  INNER JOIN fct_swap_roles_out so 
    ON p.start_date = so.start_date AND p.default_employee_key = so.swapped_out_operator_key
  UNION DISTINCT
  SELECT p.start_date, p.cell_code, so.end_ts AS bp_time
  FROM planned_cells p
  INNER JOIN fct_swap_roles_out so 
    ON p.start_date = so.start_date AND p.default_employee_key = so.swapped_out_operator_key

  -- Punkty z Absencji (domyślny operator ma L4/urlop)
  UNION DISTINCT
  SELECT p.start_date, p.cell_code, a.start_ts AS bp_time
  FROM planned_cells p
  INNER JOIN absences a 
    ON p.start_date = a.start_date AND p.default_employee_key = a.absent_operator_key
  UNION DISTINCT
  SELECT p.start_date, p.cell_code, a.end_ts AS bp_time
  FROM planned_cells p
  INNER JOIN absences a 
    ON p.start_date = a.start_date AND p.default_employee_key = a.absent_operator_key
),

-- 8. Budowanie interwałów czasowych
creating_intervals AS (
  SELECT 
    start_date,
    cell_code,
    bp_time AS interval_start,
    LEAD(bp_time) OVER (PARTITION BY start_date, cell_code ORDER BY bp_time) AS interval_end
  FROM all_time_points
  WHERE bp_time IS NOT NULL
),

valid_intervals AS (
  SELECT i.* 
  FROM creating_intervals i
  INNER JOIN planned_cells p 
    ON i.start_date = p.start_date 
    AND i.cell_code = p.cell_code
  WHERE i.interval_end IS NOT NULL 
    AND i.interval_start < i.interval_end
    AND i.interval_start >= p.plan_start 
    AND i.interval_end <= p.plan_end
),

-- 9. Ocena obsady i wyznaczanie priorytetów
evaluated_intervals AS (
  SELECT 
    i.start_date,
    CAST(date_format(i.start_date, 'yyyyMMdd') AS INT) AS date_key,
    p.line_code,
    i.cell_code,
    i.interval_start,
    i.interval_end,
    
    CASE 
      WHEN si.employee_key IS NOT NULL THEN si.employee_key
      WHEN a.absent_operator_key IS NOT NULL THEN NULL
      WHEN so.swapped_out_operator_key IS NOT NULL THEN NULL
      ELSE p.default_employee_key
    END AS assigned_operator_key,

    CASE 
      WHEN si.employee_key IS NOT NULL THEN 'SWAP'
      WHEN a.absent_operator_key IS NOT NULL THEN 'VACANT_ABSENCE'
      WHEN so.swapped_out_operator_key IS NOT NULL THEN 'VACANT_SWAPPED_OUT'
      ELSE 'DEFAULT'
    END AS assignment_type,

    ROW_NUMBER() OVER (
      PARTITION BY i.start_date, i.cell_code, i.interval_start 
      ORDER BY 
        CASE 
          WHEN si.employee_key IS NOT NULL THEN 1
          WHEN a.absent_operator_key IS NOT NULL THEN 2
          WHEN so.swapped_out_operator_key IS NOT NULL THEN 3
          ELSE 4
        END ASC
    ) AS rn
  FROM valid_intervals i
  INNER JOIN planned_cells p 
    ON i.start_date = p.start_date AND i.cell_code = p.cell_code
  LEFT JOIN fct_swap_roles_in si 
    ON i.start_date = si.start_date 
    AND i.cell_code = si.cell_code 
    AND i.interval_start >= si.start_ts 
    AND i.interval_end <= si.end_ts
  LEFT JOIN absences a 
    ON i.start_date = a.start_date 
    AND p.default_employee_key = a.absent_operator_key
    AND i.interval_start >= a.start_ts 
    AND i.interval_end <= a.end_ts
  LEFT JOIN fct_swap_roles_out so
    ON i.start_date = so.start_date
    AND p.default_employee_key = so.swapped_out_operator_key
    AND i.interval_start >= so.start_ts
    AND i.interval_end <= so.end_ts
)

SELECT 
  md5(concat_ws('||', cast(date_key as string), cell_code, cast(interval_start as string))) AS assignment_interval_key,
  date_key,
  start_date,
  line_code,
  cell_code,
  date_format(interval_start, 'HH:mm:ss') AS start_time,
  date_format(interval_end, 'HH:mm:ss') AS end_time,
  interval_start,
  interval_end,
  ROUND(timestampdiff(MINUTE, interval_start, interval_end) / 60.0, 2) AS duration_hours,
  ROUND(timestampdiff(MINUTE, interval_start, interval_end), 2) AS duration_minutes,
  assigned_operator_key,
  assignment_type,
  CURRENT_TIMESTAMP() AS _gold_created_at
FROM evaluated_intervals
WHERE rn = 1;